# Orders Optimization

This notebook isolates the optimization stage of the project for `databricks_cat.silver.orders_silver`.

## Goals
* review current Delta table layout
* create isolated benchmark copies instead of changing the source table directly
* compare managed, partitioned, and clustered layouts
* apply `OPTIMIZE` and `ZORDER` where appropriate
* rerun the same benchmark queries before and after optimization
* produce a concise final findings summary

## Optimization scenarios
* `orders_silver_perf_base` - managed Delta copy
* `orders_silver_perf_year` - managed Delta copy partitioned by `year`
* `orders_silver_perf_clustered` - managed Delta copy with seeded clustering, then `CLUSTER BY AUTO`

## Note
For clustered liquid tables, this notebook uses `OPTIMIZE` without `ZORDER`, because clustering replaces Z-ordering for that scenario.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row
import re
import time

SILVER_TABLE = "databricks_cat.silver.orders_silver"
PERF_BASE_TABLE = "databricks_cat.silver.orders_silver_perf_base"
PARTITIONED_TABLE = "databricks_cat.silver.orders_silver_perf_year"
CLUSTERED_TABLE = "databricks_cat.silver.orders_silver_perf_clustered"

print("Optimization notebook configured for:")
for table_name in [SILVER_TABLE, PERF_BASE_TABLE, PARTITIONED_TABLE, CLUSTERED_TABLE]:
    print(f" - {table_name}")

In [0]:
if not spark.catalog.tableExists(SILVER_TABLE):
    raise ValueError(f"Required table not found: {SILVER_TABLE}")

orders_df = spark.table(SILVER_TABLE)
orders_df.createOrReplaceTempView("orders_silver_v")

display(orders_df.limit(5))

## Design review

The source table is an external Delta table. To keep the project safe and reproducible, this notebook creates separate benchmark copies rather than modifying the production silver table directly.

### Why this matters
* protects the original source table
* allows side-by-side comparison of multiple layout strategies
* provides clearer benchmark evidence for the final project report

In [0]:
table_detail_cache = {}


def get_table_detail(table_name: str) -> dict:
    if table_name not in table_detail_cache:
        detail_row = spark.sql(f"DESCRIBE DETAIL {table_name}").first().asDict()
        table_detail_cache[table_name] = detail_row
    return table_detail_cache[table_name]


def build_benchmark_queries(table_name: str) -> dict:
    return {
        "date_range_filter": f"""
WITH bounds AS (
    SELECT date_add(max(to_date(order_date)), -90) AS start_date
    FROM {table_name}
)
SELECT *
FROM {table_name}
WHERE to_date(order_date) >= (SELECT start_date FROM bounds)
""",
        "product_aggregation": f"""
WITH bounds AS (
    SELECT date_add(max(to_date(order_date)), -180) AS start_date
    FROM {table_name}
)
SELECT
    product_id,
    SUM(total_amount) AS revenue,
    SUM(quantity) AS units,
    COUNT(*) AS orders
FROM {table_name}
WHERE to_date(order_date) >= (SELECT start_date FROM bounds)
GROUP BY product_id
ORDER BY revenue DESC
""",
        "customer_level_aggregation": f"""
WITH bounds AS (
    SELECT date_add(max(to_date(order_date)), -180) AS start_date
    FROM {table_name}
)
SELECT
    customer_id,
    COUNT(*) AS orders,
    SUM(total_amount) AS revenue,
    AVG(total_amount) AS avg_order_value
FROM {table_name}
WHERE to_date(order_date) >= (SELECT start_date FROM bounds)
GROUP BY customer_id
ORDER BY revenue DESC
"""
    }


STAT_PATTERNS = {
    "full": re.compile(r"statistics:\s*full", re.IGNORECASE),
    "partial": re.compile(r"statistics:\s*partial", re.IGNORECASE),
    "missing": re.compile(r"statistics:\s*missing", re.IGNORECASE)
}


def extract_plan_metrics(query_text: str) -> dict:
    plan_rows = spark.sql(f"EXPLAIN FORMATTED {query_text}").collect()
    plan_text = "\n".join(str(row[0]) for row in plan_rows)

    statistics_line_match = re.search(r"Statistics:\s*([^\n]+)", plan_text, flags=re.IGNORECASE)
    partition_filters_match = re.search(r"PartitionFilters:\s*([^\n]+)", plan_text, flags=re.IGNORECASE)
    pushed_filters_match = re.search(r"PushedFilters:\s*([^\n]+)", plan_text, flags=re.IGNORECASE)
    partition_count_match = re.search(r"PartitionCount:\s*([^\n]+)", plan_text, flags=re.IGNORECASE)
    file_index_match = re.search(r"(?:PreparedDeltaFileIndex|CatalogFileIndex)\(([^\)]*)\)", plan_text, flags=re.IGNORECASE)

    statistics_line = statistics_line_match.group(1).strip() if statistics_line_match else "not reported"

    if STAT_PATTERNS["full"].search(plan_text):
        stats_state = "full"
    elif STAT_PATTERNS["partial"].search(plan_text):
        stats_state = "partial"
    elif STAT_PATTERNS["missing"].search(plan_text):
        stats_state = "missing"
    else:
        stats_state = "not_reported"

    return {
        "statistics_line": statistics_line,
        "stats_state": stats_state,
        "partition_filters": partition_filters_match.group(1).strip() if partition_filters_match else "not reported",
        "pushed_filters": pushed_filters_match.group(1).strip() if pushed_filters_match else "not reported",
        "partition_count": partition_count_match.group(1).strip() if partition_count_match else "not reported",
        "files_scanned_hint": file_index_match.group(1).strip() if file_index_match else "not reported"
    }


def run_benchmarks(table_name: str, scenario_name: str, clear_cached_data: bool = True):
    table_detail = get_table_detail(table_name)
    benchmark_rows = []

    for query_name, query_text in build_benchmark_queries(table_name).items():
        if clear_cached_data:
            spark.catalog.clearCache()

        start_time = time.perf_counter()
        result_df = spark.sql(query_text)
        result_rows = result_df.count()
        elapsed_seconds = round(time.perf_counter() - start_time, 4)

        plan_metrics = extract_plan_metrics(query_text)
        benchmark_rows.append(Row(
            scenario=scenario_name,
            table_name=table_name,
            query_name=query_name,
            elapsed_seconds=elapsed_seconds,
            result_rows=result_rows,
            table_num_files=table_detail.get("numFiles"),
            table_size_bytes=table_detail.get("sizeInBytes"),
            partition_columns=", ".join(table_detail.get("partitionColumns") or []) or "(none)",
            clustering_columns=", ".join(table_detail.get("clusteringColumns") or []) or "(none)",
            statistics_state=plan_metrics["stats_state"],
            statistics_line=plan_metrics["statistics_line"],
            partition_filters=plan_metrics["partition_filters"],
            pushed_filters=plan_metrics["pushed_filters"],
            partitions_read_hint=plan_metrics["partition_count"],
            files_scanned_hint=plan_metrics["files_scanned_hint"]
        ))

    return spark.createDataFrame(benchmark_rows)


print("Optimization benchmark helpers are ready.")

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}"))

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_cat.silver.orders_silver_perf_base
USING DELTA
AS
SELECT *
FROM databricks_cat.silver.orders_silver;

CREATE OR REPLACE TABLE databricks_cat.silver.orders_silver_perf_year
USING DELTA
PARTITIONED BY (year)
AS
SELECT *
FROM databricks_cat.silver.orders_silver;

CREATE OR REPLACE TABLE databricks_cat.silver.orders_silver_perf_clustered
USING DELTA
CLUSTER BY (order_date, customer_id, product_id)
AS
SELECT *
FROM databricks_cat.silver.orders_silver;

ALTER TABLE databricks_cat.silver.orders_silver_perf_clustered CLUSTER BY AUTO;

## Selective caching note

Caching is used only for interactive convenience, not as the main optimization technique. The timed benchmark function clears cache before each run so layout improvements are measured more fairly.

In [0]:
eda_hot_df = orders_df.select(
    "order_date",
    "year",
    "customer_id",
    "product_id",
    "quantity",
    "total_amount"
).cache()

_ = eda_hot_df.count()
display(eda_hot_df.orderBy(F.desc("order_date")).limit(5))

In [0]:
baseline_results_df = run_benchmarks(SILVER_TABLE, "source_external_delta")
managed_base_before_df = run_benchmarks(PERF_BASE_TABLE, "managed_copy_before_optimize")
partitioned_before_df = run_benchmarks(PARTITIONED_TABLE, "partitioned_by_year_before_optimize")
clustered_before_df = run_benchmarks(CLUSTERED_TABLE, "clustered_auto_before_optimize")

before_optimize_df = (
    baseline_results_df
    .unionByName(managed_base_before_df)
    .unionByName(partitioned_before_df)
    .unionByName(clustered_before_df)
)

before_optimize_df.createOrReplaceTempView("benchmark_results_before_optimize")
display(before_optimize_df.orderBy("query_name", "scenario"))

In [0]:
%sql
OPTIMIZE databricks_cat.silver.orders_silver_perf_base
ZORDER BY (customer_id, product_id, order_date);

OPTIMIZE databricks_cat.silver.orders_silver_perf_year
ZORDER BY (customer_id, product_id, order_date);

OPTIMIZE databricks_cat.silver.orders_silver_perf_clustered;

In [0]:
managed_base_after_df = run_benchmarks(PERF_BASE_TABLE, "managed_copy_after_zorder")
partitioned_after_df = run_benchmarks(PARTITIONED_TABLE, "partitioned_by_year_after_zorder")
clustered_after_df = run_benchmarks(CLUSTERED_TABLE, "clustered_auto_after_optimize")

all_results_df = (
    baseline_results_df
    .unionByName(managed_base_before_df)
    .unionByName(partitioned_before_df)
    .unionByName(clustered_before_df)
    .unionByName(managed_base_after_df)
    .unionByName(partitioned_after_df)
    .unionByName(clustered_after_df)
)

all_results_df.createOrReplaceTempView("benchmark_results_all")
display(all_results_df.orderBy("query_name", "scenario"))

In [0]:
source_reference_df = baseline_results_df.select(
    "query_name",
    F.col("elapsed_seconds").alias("source_external_delta_seconds")
)

comparison_summary_df = (
    all_results_df
    .join(source_reference_df, on="query_name", how="left")
    .withColumn(
        "improvement_vs_source_seconds",
        F.round(F.col("source_external_delta_seconds") - F.col("elapsed_seconds"), 4)
    )
    .withColumn(
        "improvement_vs_source_pct",
        F.when(
            F.col("source_external_delta_seconds") > 0,
            F.round(
                ((F.col("source_external_delta_seconds") - F.col("elapsed_seconds")) / F.col("source_external_delta_seconds")) * 100,
                2
            )
        )
    )
)

display(comparison_summary_df.orderBy("query_name", "elapsed_seconds"))

In [0]:
profile_row = orders_df.agg(
    F.count("*").alias("row_count"),
    F.min("order_date").alias("min_order_date"),
    F.max("order_date").alias("max_order_date"),
    F.countDistinct("customer_id").alias("distinct_customers"),
    F.countDistinct("product_id").alias("distinct_products")
).first()

best_rows = (
    comparison_summary_df
    .filter(F.col("scenario") != "source_external_delta")
    .groupBy("query_name")
    .agg(F.min("elapsed_seconds").alias("best_elapsed_seconds"))
    .collect()
)

best_lookup = {row["query_name"]: row["best_elapsed_seconds"] for row in best_rows}
source_lookup = {row["query_name"]: row["source_external_delta_seconds"] for row in source_reference_df.collect()}

summary_lines = [
    "Optimization findings:",
    f"- Source table rows: {profile_row['row_count']:,}",
    f"- Date range: {profile_row['min_order_date']} to {profile_row['max_order_date']}",
    f"- Distinct customers: {profile_row['distinct_customers']:,}",
    f"- Distinct products: {profile_row['distinct_products']:,}",
    "- Scenarios tested: external Delta source, managed copy, year partitioning, and clustered auto optimization.",
    "- Optimization evidence combines runtime comparison with table layout metadata and explain-plan hints."
]

for query_name in ["date_range_filter", "product_aggregation", "customer_level_aggregation"]:
    source_time = source_lookup.get(query_name)
    best_time = best_lookup.get(query_name)
    if source_time is not None and best_time is not None and source_time > 0:
        improvement_pct = round(((source_time - best_time) / source_time) * 100, 2)
        summary_lines.append(
            f"- Best result for {query_name}: {best_time:.4f}s versus source {source_time:.4f}s ({improvement_pct}% improvement)."
        )

print("\n".join(summary_lines))

eda_hot_df.unpersist()